# Cadastro PJ por proposta e confirmação

Demonstração local da fase 2. A consulta abaixo é uma **fixture manual**: não chama Minha Receita nem LLM. O agente real usa as ferramentas `consultar_cnpj_minha_receita` e `propor_cadastro_cliente`; a confirmação fica no comando `/confirmar-cliente ID`.


In [ ]:
import sys, tempfile
from pathlib import Path

projeto = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(projeto / 'src'))
from secretaria.banco import Banco, Contexto
from secretaria.agenda import Agenda, agora_utc

temporario = tempfile.TemporaryDirectory()
caminho = Path(temporario.name) / 'secretaria.sqlite3'
banco = Banco(caminho)
banco.preparar_demo()
ctx = Contexto('demo', 'ana')
agenda = Agenda(banco)
print('Versão SQLite:', banco.con.execute('PRAGMA user_version').fetchone()[0])


In [ ]:
fixture = {'origem': 'fixture_local', 'cnpj': '33683111000280',
           'razao_social': 'Empresa Exemplo — fixture', 'nome_fantasia': 'Exemplo',
           'consultado_em': agora_utc().isoformat()}
consulta_id = agenda.registrar_consulta_cnpj(ctx, fixture)
proposta = agenda.propor_cadastro_cliente(ctx, consulta_id)
assert proposta['estado'] == 'pendente'
assert banco.con.execute('SELECT COUNT(*) FROM clientes').fetchone()[0] == 0
print(proposta)
print('Clientes antes da confirmação: 0')


In [ ]:
confirmacao = agenda.confirmar_cadastro_cliente(ctx, proposta['proposta_id'])
repeticao = agenda.confirmar_cadastro_cliente(ctx, proposta['proposta_id'])
assert confirmacao['cliente_id'] == repeticao['cliente_id']
assert repeticao['estado'] == 'ja_confirmada'
banco.fechar()
with Banco(caminho) as reaberto:
    cliente = reaberto.con.execute('SELECT nome,cnpj,origem_cadastro FROM clientes WHERE id=?',
                                   (confirmacao['cliente_id'],)).fetchone()
    auditoria = reaberto.con.execute('SELECT COUNT(*) FROM auditoria_clientes').fetchone()[0]
    assert cliente['origem_cadastro'] == 'fixture_local' and auditoria == 1
    print('Cliente após reabrir:', dict(cliente), '| auditorias:', auditoria)
temporario.cleanup()
